# Model improvement

## Dataset with Additional Categorical Features
Use the dataset containing both numerical and additional categorical features "df_final_all_features.csv" and compare it to the baseline. Log to MLflow.

In [ ]:
import os
import pathlib
import pandas as pd
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# MLflow Setup
# Point reliably to mlflow.db in the project root
# (Finds the parent directory of the current 'notebooks' folder)
project_root = pathlib.Path().resolve().parent
db_path = project_root / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{db_path}")  #use SQLite database
mlflow.set_experiment("LNP_Encapsulation_Optimization")

print(f"Tracking URI set to: sqlite:///{db_path}")

# ==========================================
# MODEL 1: THE BASELINE (Fewer Features)
# ==========================================
df_base = pd.read_csv("../data/processed/df_final_num_cat.csv") 
X_base = df_base.drop(columns=["encapsulation_efficiency"])
y_base = df_base["encapsulation_efficiency"]

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(X_base, y_base, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="RF_Baseline"):
    rf_base = RandomForestRegressor(random_state=42)
    rf_base.fit(X_train_base, y_train_base)
    
    y_pred_base = rf_base.predict(X_test_base)
    r2_base = r2_score(y_test_base, y_pred_base)
    mse_base = mean_squared_error(y_test_base,y_pred_base)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("dataset", "df_final_num_cat")
    mlflow.log_param("n_features", X_train_base.shape[1])
    mlflow.log_metric("r2", r2_base)
    mlflow.log_metric("mse", mse_base)
    
    print(f"Baseline Model - R2 Score: {r2_base:.4f} | MSE: {mse_base:.4f}")

# ==========================================
# MODEL 2: THE NEW 44-FEATURE DATASET
# ==========================================
df_new = pd.read_csv("../data/processed/df_final_all_features.csv")
X_new = df_new.drop(columns=["encapsulation_efficiency"])
y_new = df_new["encapsulation_efficiency"]

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="RF_Advanced_44_Features"):
    rf_new = RandomForestRegressor(random_state=42)
    rf_new.fit(X_train_new, y_train_new)
    
    y_pred_new = rf_new.predict(X_test_new)
    r2_new = r2_score(y_test_new, y_pred_new)
    mse_new = mean_squared_error(y_test_new, y_pred_new)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("dataset", "df_final_all_features")
    mlflow.log_param("n_features", X_train_new.shape[1])
    mlflow.log_metric("r2", r2_new)
    mlflow.log_metric("mse";mse_new)
    
    print(f"Advanced Model - R2 Score: {r2_new:.4f} | MSE: {mse_new:.4f}")

    # Plot Feature Importance for the new model
    importances = rf_new.feature_importances_
    importance_df = pd.DataFrame({"Feature": X_new.columns, "Importance": importances})
    importance_df = importance_df.sort_values(by="Importance", ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(x="Importance", y="Feature", data=importance_df, palette="viridis")
    plt.title("Top 15 Feature Importances (44-Feature Model)")
    plt.tight_layout()
    plt.savefig("feature_importance_advanced.png")
    mlflow.log_artifact("feature_importance_advanced.png") #log the saved plot to MLflow as an artifact
    plt.show()

Once you run this cell, run mlflow ui in your terminal and open [http://127.0.0.1:5000] in your browser. You will see both runs sitting right next to each other. You can even check the box next to both of them and click "Compare" to see exactly how the metrics differ.

- tuning of hyperparameters of RF 
e.g. RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42
)
- try XGBoost or LightGBM
- feature importance analysis
